In [ ]:
using BSON, Dates, DelimitedFiles, Downloads, CUDA, cuDNN, Flux, Printf, Plots, JLD2
using Flux.Zygote
include("traindata.jl")
include("neural.jl");

In [ ]:
ρ_profiles = load("InhData.jld2")["ρ_profiles"] #inhomgeneous radial density
c1_profiles = load("InhData.jld2")["c1_profiles"] #Eq. (11)
rs = load("InhData.jld2")["rs"]
ρ_windows_rad, c1_values_rad, Rr_vals_rad = generate_inout(ρ_profiles, c1_profiles, rs);

In [ ]:
ρ_profiles_plan = load("PlanData.jld2")["ρ_profiles"] #planar density
c1_profiles_plan = load("PlanData.jld2")["c1_profiles"]
r_plan = rand([-1, 1], length(ρ_profiles_plan))
ρ_windows_plan, c1_values_plan, Rr_vals_plan = generate_inout_plan(ρ_profiles_plan, c1_profiles_plan,r_plan);

In [ ]:
EOS = load("EOS.jld2")["EOS"]; #constant density
ρ_profiles_c, c1_profiles_c, r_profiles_c = read_sim_data_const(EOS)
ρ_windows_c, c1_values_c, Rr_vals_c = generate_inout(ρ_profiles_c, c1_profiles_c, r_profiles_c);

In [ ]:
ρ_windows = hcat(ρ_windows_rad,ρ_windows_plan,ρ_windows_c)
c1_values = hcat(c1_values_rad,c1_values_plan,c1_values_c)
Rr_vals = hcat(Rr_vals_rad,Rr_vals_plan,Rr_vals_c)
traindata = vcat(ρ_windows, Rr_vals);
#input: preprocessed density windows fρ (ρ_windows), normalized radial coordinate(Rr_vals)
#output: c1-values(c1_values)

In [ ]:
#set up model
model = Chain(
    Dense(size(vcat(ρ_windows, Rr_vals))[1] => 256, softplus),
    Dense(256 => 128, softplus),
    Dense(128 => 128, softplus),
    Dense(128 => 64, softplus),
    Dense(64 => 32, softplus),
    Dense(32 => 1) 
) |> gpu

display(model)  

opt = Flux.setup(Adam(), model) 

loader = Flux.DataLoader((traindata, c1_values), batchsize=256, shuffle=true)

loss(m, x, y) = Flux.mse(m(x), y)  
metric(m, x, y) = Flux.mae(m(x), y)
get_learning_rate(epoch; initial=0.0001, rate=0.03, wait=5) = epoch < wait ? initial : initial * (1 - rate)^(epoch - wait)

model_savefile = "model_rad_HS(new).bson"
println("Saving model to $(model_savefile)")
#training loop
for epoch in 1:250
    learning_rate = get_learning_rate(epoch)
    Flux.adjust!(opt, learning_rate)
    Flux.train!(loss, model, loader |> gpu, opt)
    println("completed: step $(epoch) of $(steps)")
    BSON.@save model_savefile model=cpu(model)
end

model